# Multi-Task Social NLP using Pretrained Transformers (pysentimiento)

**Project Type:** Applied Deep Learning (Transfer Learning / NLP)
**Library:** [pysentimiento](https://github.com/pysentimiento/pysentimiento)
**Author:** _(add your name)_
**Date:** _(add date)_

## Objective
This project demonstrates the use of pretrained transformer models (RoBERTa/BERTweet-based)
for multiple Social NLP tasks without training any model from scratch:

1. **Sentiment Analysis** — positive / negative / neutral
2. **Emotion Detection** — joy, sadness, fear, anger, disgust, surprise, others
3. **Irony Detection** — ironic / not ironic
4. **Hate Speech Detection** — hateful, targeted, aggressive (multi-label)

We also **evaluate the hate speech model against a real, human-labeled Kaggle dataset**
to measure how well a pretrained model performs on unseen, real-world data — this gives
us actual accuracy/precision/recall numbers rather than just qualitative examples.

## Why this counts as Deep Learning
All four tasks are powered by fine-tuned Transformer models (RoBERTa / BERTweet architectures).
This project focuses on **transfer learning** — using models already trained on large social
media corpora — rather than training a network from scratch, which is a standard and
practical real-world deep learning workflow.


## 1. Setup & Installation

In [ ]:
!pip install -q pysentimiento
!pip install -q kaggle scikit-learn seaborn matplotlib pandas


In [ ]:
from pysentimiento import create_analyzer
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', 100)


## 2. Load Pretrained Analyzers

We load each model **once** and reuse it, rather than reloading on every prediction
(reloading a transformer model per call is slow and wasteful).

In [ ]:
sentiment_analyzer = create_analyzer(task="sentiment", lang="en")
emotion_analyzer   = create_analyzer(task="emotion", lang="en")
irony_analyzer     = create_analyzer(task="irony", lang="en")
hate_analyzer      = create_analyzer(task="hate_speech", lang="en")

print("All analyzers loaded successfully.")


## 3. Single-Sentence Demonstrations

In [ ]:
sample = "I really appreciate how well this turned out, great job!"

print("Sentiment:", sentiment_analyzer.predict(sample).output)
print("Emotion:  ", emotion_analyzer.predict(sample).output)
print("Irony:    ", irony_analyzer.predict(sample).output)
print("Hate:     ", hate_analyzer.predict(sample).output)


## 4. Batch Analysis on Multiple Sentences

Testing on a variety of sentence types (positive, negative, fearful, ironic, aggressive)
to see how the models behave across different inputs.

In [ ]:
test_sentences = [
    "yayyy, we finally finished the project!",
    "Scared of swimming in deep water.",
    "Oh great, another Monday. Just what I needed.",   # ironic/sarcastic
    "I really appreciate your help today.",
    "This is the worst experience I've ever had.",
    "I want to hurt you.",
    "The weather is nice today.",
    "You are such an amazing friend, thank you!",
]

def analyze_text(txt):
    return {
        "text": txt,
        "sentiment": sentiment_analyzer.predict(txt).output,
        "emotion": emotion_analyzer.predict(txt).output,
        "irony": irony_analyzer.predict(txt).output,
        "hate_labels": hate_analyzer.predict(txt).output,
    }

results = [analyze_text(t) for t in test_sentences]
results_df = pd.DataFrame(results)
results_df


## 5. Visualizing Emotion Distribution Across Test Sentences

In [ ]:
emotion_counts = results_df['emotion'].value_counts()

plt.figure(figsize=(7,4))
sns.barplot(x=emotion_counts.index, y=emotion_counts.values, palette="viridis")
plt.title("Emotion Distribution Across Test Sentences")
plt.xlabel("Emotion")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 6. Evaluating Hate Speech Detection on a Real Labeled Dataset

Qualitative examples are useful, but a real project needs **quantitative evaluation**.
Here we test the pretrained hate speech model against the
[Hate Speech and Offensive Language Dataset](https://www.kaggle.com/datasets/mrmorj/hate-speech-and-offensive-language-dataset)
from Kaggle, which contains ~25,000 tweets labeled by humans as:

- `0` = hate speech
- `1` = offensive language
- `2` = neither

We convert this to a **binary problem** (hateful vs. not hateful) to match pysentimiento's
`hateful` label, then compare model predictions against the ground-truth human labels.

### 6.1 Download the dataset from Kaggle

Upload your `kaggle.json` API key when prompted (Kaggle → Account → Create New Token).

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d mrmorj/hate-speech-and-offensive-language-dataset
!unzip -q -o hate-speech-and-offensive-language-dataset.zip
!ls


In [ ]:
# Adjust filename below if it differs after unzip (check the !ls output above)
df = pd.read_csv('labeled_data.csv')
print(df.shape)
df.head()


### 6.2 Prepare a Sample for Evaluation

Running the transformer model on all 25,000 rows would take a while, so for this
demo we evaluate on a random sample of 100 tweets — still statistically meaningful
for a beginner project, and fast enough to run in a few minutes.

In [ ]:
sample_df = df.sample(100, random_state=42).reset_index(drop=True)

# Ground truth: 1 = hateful (original class 0), 0 = not hateful (original class 1 or 2)
sample_df['true_label'] = sample_df['class'].apply(lambda x: 1 if x == 0 else 0)

sample_df[['tweet', 'class', 'true_label']].head()


In [ ]:
def predict_hateful(text):
    result = hate_analyzer.predict(text)
    return 1 if 'hateful' in result.output else 0

sample_df['predicted_label'] = sample_df['tweet'].apply(predict_hateful)
sample_df[['tweet', 'true_label', 'predicted_label']].head(10)


### 6.3 Accuracy, Precision, Recall & Confusion Matrix

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

acc = accuracy_score(sample_df['true_label'], sample_df['predicted_label'])
print(f"Accuracy on 100-tweet sample: {acc*100:.2f}%\n")

print(classification_report(sample_df['true_label'], sample_df['predicted_label'],
                             target_names=['Not Hateful', 'Hateful']))


In [ ]:
cm = confusion_matrix(sample_df['true_label'], sample_df['predicted_label'])

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Hateful', 'Hateful'],
            yticklabels=['Not Hateful', 'Hateful'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix: Pretrained Model vs. Human Labels')
plt.tight_layout()
plt.show()


## 7. Limitations & Ethical Considerations

- **Label bias**: The Kaggle hate speech dataset was labeled by human annotators, whose
  judgments may reflect their own cultural or personal biases, especially around slang
  and reclaimed language used differently across communities.
- **Domain mismatch**: The pretrained model was trained on different social media data,
  so its notion of "hateful" may not perfectly match this dataset's annotation guidelines
  — this is a likely cause of any accuracy gap seen above.
- **Binary simplification**: Collapsing "offensive" and "neither" into a single
  "not hateful" class loses nuance — a more advanced version of this project could
  treat it as a 3-class problem instead.
- **Not production-ready**: This model should not be used as the sole moderation system
  for a real platform without further fine-tuning, human review, and fairness auditing.

## 8. Conclusion

This project demonstrated **transfer learning** for Social NLP by using pretrained
Transformer models (RoBERTa/BERTweet architectures via `pysentimiento`) across four
tasks: sentiment analysis, emotion detection, irony detection, and hate speech detection.

Rather than training a model from scratch, we evaluated an existing pretrained model
against a real, independently human-labeled dataset, measuring actual accuracy,
precision, recall, and visualizing results with a confusion matrix. This mirrors a
common real-world deep learning workflow: **using pretrained models and validating
them on your own data**, rather than always training from zero.

**Possible extensions:**
- Evaluate on the full dataset instead of a 100-row sample
- Compare pysentimiento against a custom-trained LSTM/CNN classifier (see baseline notebook)
- Try the Spanish/Portuguese/Italian models to test multilingual performance
